--- SQL --- PYTHON --- PYSPARK ---

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
#1. How to find out Duplicates while loading/ingesting employees data and take into consider only latest record based on recently modified (Based on last_modified_date, updated_date, loaded_date or any date column from table)

emp_data = [
    ["0001","10","akshara","M","30000","2024-01-03"],
    ["0002","20","sadhana","F","40000","2023-11-29"],
    ["0003","20","suraksha","F","32000","2023-01-07"],
    ["0004","30","arjun","M","24000","2024-10-09"],
    ["0005","10","dharma","M","3000","2022-09-15"],
    ["0006","30","kousalya","F","2000","2021-02-14"],
    ["0007","10","Sumitra","F","1000","2020-07-22"],
    ["0008","30","Nakula","M","50000","2022-11-19"],
    ["0008","30","Nakula",None,"50000","2022-11-20"]
]
emp_schema="empid string, deptid string, empname string, gender string, salary string, hire_date string"

df_emp = spark.createDataFrame(emp_data, emp_schema)
df_emp.display()

df_emp=df_emp.withColumn('hire_date',col('hire_date').cast(DateType()))
df_emp=df_emp.orderBy('empid','hire_date',ascending = [1,0]).dropDuplicates(subset=['empid'])
df_emp.display()
df_emp.printSchema()

In [0]:
# 2. how to merge same data coming from multiple file sources and have some not consistent?
df = spark.read.format('parquet')\
                .option('mergeSchema',True)
                .load('File/Data/datafiles')


In [0]:
# 3. difference between Spark and Hadoop MapReduce as performance and scalability wise?
# 1. spark uses in-memory data process whareas hadoop not.
# 2. spark also uses to write data when we need or if intermediate data not fit in memory.
# 3. spark is faster due to query optimization.
# 4. spark is meant for both batch and stream/real time process whereas hadoop is only for batch.

In [0]:
#4. You are working with a real-time data pipeline, and you notice missing values in your streaming data Column - Category. How would you handle null or missing values in such a scenario?**

df_emp_stream = spark.readStream.schema("empid int, value string").csv("source/hr/emp/stream")
df_emp_stream = df_emp_stream.fillNa({'comm':'NA'})


In [0]:
# 5. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this information?
user_data = [("user1", 5), ("user2", 8), ("user3", 2), ("user4", 10), ("user2", 3)]
user_columns = ["user_id", "actions"]

df_user = spark.createDataFrame(user_data, user_columns)
#df_user.display()

df_user = df_user.groupBy('user_id').agg(sum('actions').alias('total_actions')).orderBy('total_actions',ascending=False).limit(5)

df_user.display()

In [0]:
#6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?

from pyspark.sql.window import Window

customer_data = [("cust1", "2023-12-01", 100), ("cust2", "2023-12-02", 150),
        ("cust1", "2023-12-03", 200), ("cust2", "2023-12-04", 250)]
customer_columns = ["customer_id", "transaction_date", "sales"]
customer_df = spark.createDataFrame(customer_data, customer_columns)
#customer_df.display()
#df_emp=df_emp.withColumn('hire_date',col('hire_date').cast(DateType()))
customer_df = customer_df.withColumn('transaction_date',col('transaction_date').cast(DateType()))
customer_df = customer_df.withColumn('rk',dense_rank().over(Window.partitionBy('customer_id').orderBy(col('transaction_date').desc()))).filter(col('rk')==1)
customer_df.display()

In [0]:
#7. You need to identify customers who haven’t made any purchases in the last 30 days. How would you filter such customers?**
data = [("cust1", "2025-12-01"), ("cust2", "2024-11-20"), ("cust3", "2024-11-25")]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)
#df.display()

df = df.withColumn('last_purchase_date',to_date('last_purchase_date'))
df = df.withColumn('diff',datediff(current_date(),'last_purchase_date')).filter(col('diff')>30)
df.display()





In [0]:
#8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?**
data = [("customer1", "The product is great"), ("customer2", "Great product, fast delivery"), ("customer3", "Not bad, could be better")]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

#df.display()

df_split = df.withColumn('feedback',split('feedback',' '))
#df_split.display()

df_split_Explode = df_split.withColumn('feedback',explode('feedback'))
#df_split_Explode.display()
df_split_grp_cnt= df_split_Explode.withColumn('feedback',lower('feedback')).groupBy('feedback').agg(count('feedback').alias('word_count'))
df_split_grp_cnt.display()

In [0]:
# 9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?
from pyspark.sql.window import Window
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-02", 200),
        ("product1", "2023-12-03", 150), ("product2", "2023-12-04", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.printSchema()
#df.display()
df = df.withColumn('date',to_date('date'))
df.printSchema()
df_cum_sum = df.withColumn('cum_sum',sum('sales').over(Window.partitionBy('product_id').orderBy('date')))
df_cum_sum.display()

In [0]:
# 10. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?**
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
#df.display()
df_deduplicate = df.withColumn('rk',row_number().over(Window.partitionBy('name').orderBy('age'))).filter(col('rk')==1)
df_deduplicate.display()

In [0]:
# 11. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?**
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60), 
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

#df.display()
df_avg_duration_by_user= df.withColumn('avg_duration_for_user',avg('duration').over(Window.partitionBy('user_id').orderBy('user_id')))
df_avg_duration_by_user.display()
df_avg = df.groupBy('user_id').agg(avg('duration').alias('avg_duration_time_per_user'))
df_avg.display()

In [0]:
# **12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?**
from pyspark.sql.window import Window
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150), 
        ("product1", "2023-12-02", 200), ("product2", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
#df.display()
#df.printSchema()
df = df.withColumn('date',to_date('date'))
#df.display()
#df.printSchema()
df = df.withColumn('date',month('date')).groupBy('date','product_id').agg(sum('sales').alias('sales'))
df = df.withColumn('rk',dense_rank().over(Window.partitionBy('date').orderBy(col('sales').desc()))).filter(col('rk')==1)
df.display()

In [0]:
# what is role of SparkContext in Pyspark?
# explain spark architecture?

In [0]:
#**13. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions, and sometimes updates can cause inconsistent reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?**
df = spark.read.format('parquet').load('path')

from delta.tables import DeltaTable
delta_tbl = DeltaTable.forPath('path')

delta_tbl.alias('trg').merge(df.alias('src'),"src.id == trg.id")\
		.whenNotMatchedInsertAll()\
		.whenMatchedUpdateAll()\
		.execute()


In [0]:
# 14. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (Almost). How would you do this?**
df = spark.read.format('parquet')\
		.option('inferSchema',True)\
		.load('path')

In [0]:
##**15. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?**
df = spark.read.format("csv")\
			.option("mode","DROPMALFORMED")\
			.load("stage_location_path")

In [0]:
# Q. Difference between RDD and DataFrame and Dataset?
# Q. what is QUERY OPTIMIZATION?
# Q. Explain SPARK SESSION?
# Q. Diff Between WIDE TRANSFORMATIONS AND NARROW TRANSFORMATIONS?
# Q. What is USE of COALESCE() and REPARTITION()?
# Q. Diff Between CACHE() and PERSIST()?
# Q. What is importance of PARTITIONS IN PYSPARK?


In [0]:
# 16. You have a dataset containing the names of employees and their departments. You need to find the department with the most employees.**
data = [("Alice", "HR"), ("Bob", "Finance"), ("Charlie", "HR"), ("David", "Engineering"), ("Eve", "Finance")]
columns = ["employee_name", "department"]

df = spark.createDataFrame(data, columns)
#df.display()

df = df.groupBy('department').agg(count('employee_name').alias('emp_cnt')).sort('emp_cnt',ascending=False)
df.display()

In [0]:
# 17 While processing sales data, you need to classify each transaction as either 'High' or 'Low' based on its amount. How would you achieve this using a when condition**
data = [("product1", 100), ("product2", 300), ("product3", 50)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
# df.display()

df = df.withColumn('price_cat',when(col('sales')>50,"High").otherwise("Low"))
df.display()


In [0]:
# 18. While analyzing a large dataset, you need to create a new column that holds a timestamp of when the record was processed. How would you implement this and what can be the best USE CASE?**
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
#df.display()
df = df.withColumn('processed_time',current_timestamp())
df.display()

In [0]:
# 19. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it. How would you achieve this?**
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
#df.display()
df.createOrReplaceTempView('v_products')
view_output = spark.sql('select * from v_products')
view_output.display()

# or run in SQL worksheet directly


In [0]:
%sql
select * from v_products;

In [0]:
# 20. 
df.createOrReplaceGlobalTempView('v_g_products')
df_output = spark.sql('select * from global_temp.v_g_products')

df_output.display()



In [0]:
# 21. You need to query data from a PySpark DataFrame using SQL, but the data includes a nested structure. How would you flatten the data for easier querying?**
data = [("product1", {"price": 100, "quantity": 2}), 
        ("product2", {"price": 200, "quantity": 3})]
columns = ["product_id", "product_info"]

df = spark.createDataFrame(data, columns)
#df.display()

#df.select('product_id','product_info.price','product_info.quantity').display()

#op = df.select("product_id","product_info.price","product_info.quantity")

df.select("product_id","product_info.price","product_info.quantity").createOrReplaceGlobalTempView('prod_t')

op = spark.sql('select * from global_temp.prod_t')
op.display()

In [0]:
# 22. You are ingesting data from an external API in JSON format where the schema is inconsistent. How would you handle this situation to ensure a robust pipeline?**

df = spark.read.format("json").option("mergeSchema",True)

In [0]:
# 23  While reading data from Parquet, you need to optimize performance by partitioning the data based on a column. How would you implement this?
df.write.format(("parquet").mode("append").partitionBy("category").save("location")

In [0]:
#24. You are working with a large dataset in Parquet format and need to ensure that the data is written in an optimized manner with proper compression. How would you accomplish this?
df.write.format("parquet").option("compression","snappy")

In [0]:
#31. Your company uses a large-scale data pipeline that reads from Delta tables and processes data using complex aggregations. However, performance is becoming an issue due to the growing dataset size. How would you optimize the performance of the pipeline?

%sql
OPTIMIZE tabledelta ZORDER BY ('order_date')

#Q. what OPTIMIZE will do?

#Q. what ZORDER BY will do?

In [0]:
#Q. what are broadcast variables, and why are they used?
#Q. what is difference between df.show() and df.collect()?
#Q. what is LAZY EVALUATION in PYSPARK?
#Q. what are the advantages of Delta Lake over traditional file formats?
#Q. What happens when a PySpark job runs out of memory?
#Q. what is AQE in PySpark? Why is it useful?--ADAPTIVE QUERY EXECUTIONS.--SPARK OPTIMIZATION.
#--DYNAMIC PARTIITON.
#--JOIN STRATEGIES OPTIMIZATIONS.
#--DYNAMICALLY OPTIMIZES SKEWNESS.
#Q. How would you handle skewed data in PySpark?
#Q. What is broadcast join and when should you use it?
#Q. What is spill in Spark, and why does it happen?
#Q. What are Delta Lake's time travel features and how do they work?
#--versions.
#--describe history emp_table;
#--restore emp_table to version as of 2;

In [0]:
#You are processing sales data. Group by product categories and create a list of all product names in each category.**
data = [("Electronics", "Laptop"), ("Electronics", "Smartphone"), ("Furniture", "Chair"), ("Furniture", "Table")]
columns = ["category", "product"]
df = spark.createDataFrame(data, columns)
#df.display()

df = df.groupBy('category').agg(collect_list('product').alias('product'))
df.display()

In [0]:
#You are analyzing orders. Group by customer IDs and list all unique product IDs each customer purchased.**
data = [(101, "P001"), (101, "P002"), (102, "P001"), (101, "P001")]
columns = ["customer_id", "product_id"]
df = spark.createDataFrame(data, columns)
#df.display()
df = df.groupBy('customer_id').agg(collect_set('product_id').alias('unique_products'))
df.display()

In [0]:
# For customer records, combine first and last names only if the email address exists.**
data = [("John", "Doe", "john.doe@example.com"), ("Jane", "Smith", None)]
columns = ["first_name", "last_name", "email"]
df = spark.createDataFrame(data, columns)
#df.display()
df = df.withColumn("fullname",when(col('email').isNotNull(),concat_ws('-',col('first_name'),col('last_name'))).otherwise(None))
df.display()

In [0]:
# You have a DataFrame containing customer IDs and a list of their purchased product IDs. Calculate the number of products each customer has purchased.**
data = [
    (1, ["prod1", "prod2", "prod3"]),
    (2, ["prod4"]),
    (3, ["prod5", "prod6"]),
]
myschema = "customer_id INT ,product_ids array<STRING>"

df = spark.createDataFrame(data, myschema)
#df.display()

df = df.withColumn("number_of_products",size(col('product_ids')))
df.display()

In [0]:
# You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes.
data = [
    ("1",),
    ("123",),
    ("4567",),
]
schema = ["employee_id"]

df = spark.createDataFrame(data, schema)
#df.display()
df = df.withColumn("employee_id",lpad(col('employee_id'),6,"0"))
df.display()

In [0]:
# You need to validate phone numbers by checking if they start with "91"
data = [
    ("911234567890",),
    ("811234567890",),
    ("912345678901",),
]
schema = ["phone_number"]

df = spark.createDataFrame(data, schema)
#df.display()

df = df.filter(substring(col('phone_number'),1,2) == "91")
df.display()

In [0]:
# You have a dataset with courses taken by students. Calculate the average number of courses per student.
data = [
    (1, ["Math", "Science"]),
    (2, ["History"]),
    (3, ["Art", "PE", "Biology"]),
]
schema = ["student_id", "courses"]

df = spark.createDataFrame(data, schema)
#df.display()

df = df.withColumn("course_size",size('courses')).groupBy().agg(avg('course_size'))
df.display()

In [0]:
# You have a dataset with primary and secondary contact numbers. Use the primary number if available; otherwise, use the secondary number
data = [
    (None, "1234567890"),
    ("9876543210", None),
    ("7894561230", "4567891230"),
]
schema = ["primary_contact", "secondary_contact"]

df = spark.createDataFrame(data, schema)
#df.display()
df = df.withColumn("contact",coalesce(col('primary_contact'),col('secondary_contact')))
df.display()

In [0]:
# You are categorizing product codes based on their lengths. If the length is 5, label it as "Standard"; otherwise, label it as "Custom".**
data = [
    ("prod1",),
    ("prd234",),
    ("pr9876",),
]
schema = ["product_code"]

df = spark.createDataFrame(data, schema)
# df.display()
df = df.withColumn("code_flag", when(length(col('product_code'))==5,"Standard").otherwise("Custom"))
df.display()